## Описание задания
Вы получили выгрузку заказов (orders_dirty.csv) для построения модели, предсказывающей отмену заказа (целевая переменная: status).

Однако данные «сырые»: в них есть дубли, опечатки, пропуски и логические ошибки. Также у вас есть скрытый файл «истины» (orders_truth_hidden.csv), в котором зафиксированы все реальные проблемы датасета. Этот файл нужен, чтобы проверить, насколько внимательно вы нашли ошибки.

Ваша цель:
- Написать детекторы ошибок и добиться Recall ≥ 80% (найти не менее 80% проблем из скрытого файла).
- Очистить данные и обучить модель, сравнив качество до и после.

Инструкция к выполнению

Задание выполняется в Jupyter Notebook или Google Colab. Скачайте шаблон для выполнения и файлы с данными из раздела «Инструменты, которые пригодятся для выполнения задания» и следуйте шагам ниже.


In [182]:
# импорт базовых библиотек
import numpy as np  # численные операции
import pandas as pd  # таблицы и анализ

# импорт датасета из библиотеки
from sklearn.datasets import load_breast_cancer  # учебный датасет
from sklearn.model_selection import train_test_split  # разбиение train/test
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# импорт графиков
import matplotlib.pyplot as plt  # графики
import seaborn as sns  # статистическая визуализация


### Шаг 1. Первичный анализ (EDA)
Загрузите файл orders_dirty.csv. Проведите беглый осмотр данных:
- Оцените размер таблицы и типы данных.
- Найдите топ-3 столбца с наибольшим количеством пропусков.
- Посмотрите уникальные значения в столбцах payment_method и currency. Есть ли там странные значения?
- Проверьте, есть ли дубликаты строк по идентификатору заказа.

В ноутбуке: выведите 2–3 таблицы/графика и напишите краткий вывод (что бросилось в глаза).


In [183]:
dirty_train = pd.read_csv('orders_dirty.csv')

display(dirty_train.describe())
display(dirty_train.head(5))

# размер и типы
print("Размер dirty_train:", dirty_train.shape)
print("\nТипы столбцов:")
print(dirty_train.dtypes.tail(5))  # последние 5 для компактности


,row_id,price,quantity,total_amount
count,2570.000000,2560.000000,2560.000000,2570.000000
mean,1284.500000,32.988891,2.899219,98.811393
std,742.039419,18.804387,1.467906,77.699330
min,0.000000,-10.000000,-1.000000,8.000000
25%,642.250000,20.727500,2.000000,44.665000
50%,1284.500000,29.140000,3.000000,79.080000
75%,1926.750000,40.652500,4.000000,129.940000
max,2569.000000,238.800000,5.000000,1194.000000


,row_id,order_id,customer_id,order_date,delivery_date,status,payment_method,currency,price,quantity,total_amount,channel
0,2211,ORD102211,C79689,2025-08-04,2025-08-05,delivered,paypal,EUR,47.42,4.0,189.68,web
1,2136,ORD102136,C59002,2025-08-25,2025-08-31,delivered,credit_card,EUR,39.94,3.0,119.82,app
2,1178,ORD101178,C33324,2025-01-28,2025-02-04,delivered,credit_card,EUR,29.06,NaN,116.24,web
3,1910,ORD101910,C16688,2024-03-29,2024-04-05,delivered,paypal,EUR,47.86,4.0,191.44,web
4,649,ORD100649,C86369,2024-03-20,2024-03-26,delivered,credit_card,EUR,9.31,5.0,46.55,web


Размер dirty_train: (2570, 12)

Типы столбцов:
currency            str
price           float64
quantity        float64
total_amount    float64
channel             str
dtype: object


In [184]:
# доля пропусков
print("\nТоп столбцов по пропускам (%):")
print((dirty_train.isna().mean() * 100).round(2).sort_values(ascending=False).head(10))

# доля пропусков
print("\nДубликаты в order_id:")
print(dirty_train.duplicated(subset=['order_id']).sum())

# уникальные значения категориального столбца
# print("\nУникальные currency:")
# print(dirty_train["currency"].astype(str).unique()[:15])

print("\nУникальные payment_method:")
print(dirty_train["payment_method"].astype(str).unique()[:15])

print("\nУникальные status:")
print(dirty_train["status"].astype(str).unique()[:15])


Топ столбцов по пропускам (%):
order_id          0.78
customer_id       0.78
payment_method    0.58
status            0.58
quantity          0.39
order_date        0.39
price             0.39
row_id            0.00
currency          0.00
delivery_date     0.00
dtype: float64

Дубликаты в order_id:
89

Уникальные payment_method:
<StringArray>
['paypal', 'credit_card', 'cash', 'card', 'crypto', nan, 'apple_pay']
Length: 7, dtype: str

Уникальные status:
<StringArray>
[  'delivered',  'processing',           nan,     'shipped',        'done',
   'cancelled',    'x_cancel', 'in_progress']
Length: 8, dtype: str


### Наблюдения
- В данных присутсвуют пропуски
- Отрицательные значения там где это не возможно: quantity, price
- В данных присутсвует значения nan в столбцах payment_method и status

### Шаг 2. Реализация детекторов
Вам нужно реализовать 4 функции-детектора. Каждая функция должна возвращать список найденных проблем в формате кортежа: (индекс_строки, название_колонки, тип_ошибки).

Бизнес-правила для проверки:
- Пропуски (missing): Любое значение NaN в данных недопустимо.
- Дубликаты (duplicate_row): Один order_id должен встречаться только один раз. Повторы считаются ошибкой.
Нарушения диапазонов (range_violation):
- Цена (price) и количество (quantity) не могут быть отрицательными или равными нулю.

Ошибки категорий (category_inconsistency):
- Статус заказа может быть только: delivered, shipped, cancelled, processing.
- Метод оплаты может быть только: credit_card, paypal, cash.
- Любые другие значения (включая опечатки, разный регистр) считаются ошибкой.
Подсказка: используйте множества (set) или списки допустимых значений (whitelist) для проверки категорий.

In [185]:
 # допустимые «чистые» категории
ALLOWED_PAYMENT_METHOD = {'credit_card', 'paypal', 'cash'}
ALLOWED_STATUS= { 'delivered', 'shipped', 'cancelled', 'processing'}


# детектор пропусков
def detect_missing(df):
    rows, cols = np.where(df.isna())
    return {(int(df.iloc[r]['row_id']), df.columns[c], "missing") for r, c in zip(rows, cols)}

# детектор дубликатов (берём повторные строки)
def detect_duplicates(df):
    dup_idx = df.loc[df['order_id'].duplicated(keep="first"),'row_id']
    return {(int(i), "order_id", "duplicate_row") for i in dup_idx}

# детектор нарушений диапазона
def detect_range_violations(df):
    mask_q = df["quantity"] <= 0
    mask_p = df["price"] <= 0
    result = set()
    result.update({(int(i), "quantity", "range_violation") for i in df.loc[mask_q, 'row_id']})
    result.update({(int(i), "price", "range_violation") for i in df.loc[mask_p, 'row_id']})
    return result

# детектор грязных категорий
def detect_category_inconsistency(df):
    mask_p = ~df["payment_method"].isin(ALLOWED_PAYMENT_METHOD)
    mask_s = ~df["status"].isin(ALLOWED_STATUS)
    result = set()
    result.update({(int(i), "payment_method", "category_inconsistency") for i in df.loc[mask_p, 'row_id']})
    result.update({(int(i), "status", "category_inconsistency") for i in df.loc[mask_s, 'row_id']})
    return result

In [186]:
# запускаем все детекторы
detected_set = set().union(
    detect_missing(dirty_train),
    detect_duplicates(dirty_train),
    detect_range_violations(dirty_train),
    detect_category_inconsistency(dirty_train),
)

# переводим в DataFrame
detected_df = pd.DataFrame(list(detected_set), columns=["row_id", "column", "issue_type"])
# display(detected_df)

# сколько нашли по типам
detected_df["issue_type"].value_counts().sort_index()


issue_type
category_inconsistency    100
duplicate_row              89
missing                   100
range_violation            75
Name: count, dtype: int64

## Шаг 3. Оценка качества поиска
Загрузите файл orders_truth_hidden.csv. Сравните список ошибок, который нашли ваши детекторы, с эталонным списком.

Рассчитайте метрику Recall по формуле:

Recall = Количество совпавших ошибок (TP) / Всего ошибок в файле truth (TP + FN)
Цель: получить общий Recall не ниже 80%.

In [216]:
orders_truth_hidden = pd.read_csv('orders_truth_hidden.csv')
display(orders_truth_hidden["issue_type"].value_counts().sort_index())

# формируем множества «истина» и «нашли»
true_set = set(orders_truth_hidden.itertuples(index=False, name=None))
found_set = set(detected_df.itertuples(index=False, name=None))

# считаем TP и recall
tp_set = true_set & found_set
recall = len(tp_set) / len(true_set)

# печатаем итог
print(f"Всего истинных проблем: {len(true_set)}")
print(f"Найдено корректно (TP): {len(tp_set)}")
print(f"Recall: {recall:.2%}")

issue_type
category_inconsistency     70
duplicate_row             140
missing                   100
range_violation            75
Name: count, dtype: int64

Всего истинных проблем: 385
Найдено корректно (TP): 315
Recall: 81.82%


In [188]:
# детализация recall по каждому типу проблемы
tp_df = pd.DataFrame(list(tp_set), columns=["row_id", "column", "issue_type"])

by_type = (
    orders_truth_hidden.groupby("issue_type").size().rename("true_count").to_frame()
    .join(tp_df.groupby("issue_type").size().rename("tp_count"), how="left")
    .fillna(0)
)

by_type["tp_count"] = by_type["tp_count"].astype(int)
by_type["recall"] = (by_type["tp_count"] / by_type["true_count"]).round(3)

by_type.sort_values("recall", ascending=False)

,true_count,tp_count,recall
issue_type,,,
category_inconsistency,70,70,1.0
missing,100,100,1.0
range_violation,75,75,1.0
duplicate_row,140,70,0.5


## Шаг 4. Очистка и Моделирование
Докажите бизнесу, что чистка данных имеет смысл.

Напишите функцию очистки:
- Удалите дубликаты.
- Заполните пропуски (например, медианой для чисел и модой для категорий).
- Исправьте опечатки в категориях (приведите к нижнему регистру, устраните неявные дубликаты).
- Обработайте некорректные числовые значения (например, возьмите модуль от отрицательной цены).

Обучите модель:
- Подготовьте данные: целевая метка формируется из status: target=1, если status=‘cancelled’, иначе 0
- Обучите простую LogisticRegression дважды: на «грязных» данных и на очищенных.
- Сравните метрику Accuracy

In [197]:
# функция очистки dirty_train
def clean_quality_issues(df):
    clean = df.copy()

    # 1) приводим к нижнему регистру и удаляем дубликаты
    str_cols = clean.select_dtypes(include=["str","object","string"]).columns.tolist()
    clean[str_cols] = clean[str_cols].apply(lambda x: x.str.lower().str.strip())
    clean = clean.drop_duplicates(keep="first", subset ='order_id').reset_index(drop=True)
    clean = clean.drop_duplicates(keep="first").reset_index(drop=True)

    # 2) чистим и нормализуем категории
    clean["status"] = clean["status"].astype(str).str.strip()
    cat_map = {
        "delivered": "delivered",
        "shipped": "shipped",
        "cancelled": "cancelled",
        "x_cancel": "cancelled",
        "in_progress": "processing",
        "done": "shipped",
        "processing": "processing"
    }
    clean["status"] = clean["status"].replace(cat_map)
    clean.loc[~clean["status"].isin(ALLOWED_STATUS), "status"] = np.nan
    clean["status"] = clean["status"].fillna(clean["status"].mode()[0])
    clean.loc[~clean["payment_method"].isin(ALLOWED_PAYMENT_METHOD), "payment_method"] = np.nan
    clean["payment_method"] = clean["payment_method"].fillna(clean["payment_method"].mode()[0])

    # 3) некорректный диапазон -> NaN
    clean.loc[clean["quantity"] == 0, "quantity"] = np.nan
    clean.loc[clean["price"] == 0, "price"] = np.nan
    clean["quantity"] = clean["quantity"].abs()
    clean["price"] = clean["price"].abs()

    # 4) заполняем числовые пропуски медианой
    num_cols = clean.select_dtypes(include="number").columns.tolist()
    clean[num_cols] = clean[num_cols].fillna(clean[num_cols].median())

    return clean

In [199]:
# применяем очистку
clean_train = clean_quality_issues(dirty_train)

# быстрые проверки после очистки
post_check = {
    "rows": len(clean_train),
    "missing_total": int(clean_train.isna().sum().sum()),
    "duplicate_rows": int(clean_train.duplicated().sum()),
    "range_violations_quantity": int((clean_train["quantity"] < 0).sum()),
    "range_violations_price": int((clean_train["price"] < 0).sum()),
    "bad_categories_status": int((~clean_train["status"].isin(ALLOWED_STATUS)).sum()),
    "bad_categories_payment_method": int((~clean_train["payment_method"].isin(ALLOWED_PAYMENT_METHOD)).sum())
}
pd.Series(post_check)


rows                             2481
missing_total                      31
duplicate_rows                      0
range_violations_quantity           0
range_violations_price              0
bad_categories_status               0
bad_categories_payment_method       0
dtype: int64

In [191]:
print("=========================== dirty train:")
display(dirty_train.describe())

print("=========================== clean train:")
clean_train.describe()

=========================== dirty train:


,row_id,price,quantity,total_amount
count,2570.000000,2560.000000,2560.000000,2570.000000
mean,1284.500000,32.988891,2.899219,98.811393
std,742.039419,18.804387,1.467906,77.699330
min,0.000000,-10.000000,-1.000000,8.000000
25%,642.250000,20.727500,2.000000,44.665000
50%,1284.500000,29.140000,3.000000,79.080000
75%,1926.750000,40.652500,4.000000,129.940000
max,2569.000000,238.800000,5.000000,1194.000000


=========================== clean train:


,row_id,price,quantity,total_amount
count,2481.000000,2481.000000,2481.000000,2481.000000
mean,1267.529625,33.145663,2.943974,98.928992
std,731.954981,18.427773,1.415241,77.678937
min,0.000000,1.000000,1.000000,8.000000
25%,636.000000,20.890000,2.000000,44.800000
50%,1266.000000,29.120000,3.000000,79.560000
75%,1900.000000,40.200000,4.000000,129.960000
max,2569.000000,238.800000,5.000000,1194.000000


In [192]:
# считаем ключевые статистики до и после очистки
summary = pd.DataFrame({
    "dirty_status_nunique": [dirty_train["status"].nunique()],
    "clean_status_nunique": [clean_train["status"].nunique()],
    
    "dirty_payment_method_nunique": [dirty_train["payment_method"].nunique()],
    "clean_payment_method_nunique": [clean_train["payment_method"].nunique()]
    })

summary.T.rename(columns={0: "value"})

,value
dirty_status_nunique,7
clean_status_nunique,4
dirty_payment_method_nunique,6
clean_payment_method_nunique,3


In [215]:
def get_acc(df):
    
    df_work = df.copy()
    df_work['target'] =(df_work['status'] == 'cancelled').astype(int)
    num_cols = df_work.select_dtypes(include="number").columns.tolist()
    df_work = df_work[num_cols].fillna(-1)
    
    X = df_work.drop(["target","total_amount","row_id"], axis=1)
    y = df_work["target"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, random_state=42)
    )
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))

acc_dirty = get_acc(dirty_train)
acc_clean = get_acc(clean_train)

print(f"Точность на грязных данных: {acc_dirty:.1%}")
print(f"Точность на чистых данных:  {acc_clean:.1%}")
print(f"Прирост качества:           {acc_clean - acc_dirty:+.1%}")

Точность на грязных данных: 90.7%
Точность на чистых данных:  90.2%
Прирост качества:           -0.5%


## Шаг 5. Итоги (рефлексия)
В конце ноутбука добавьте Markdown-ячейку с выводами (5–7 предложений):
- Какие ошибки было найти сложнее всего?
- Как сильно выросла точность модели после очистки?
- Какую одну проверку (из реализованных вами) вы бы поставили на ежедневный мониторинг в первую очередь и почему?



===ИТОГИ================
1. Сложным осталось duplicate_row я так и не понял почему в истенном файле их больше.
2. Разница 90.7% и 90.2% это не ухудшение качества. Это снятие маски, грязные данные создавали ложное впечатление хорошей модели. Реальная предсказательная способность модели около 90.2%, и теперь ей можно доверять.
3. Внедрил бы проверку или даже защиту на создание дублирующиих order_id

## Перед сдачей задания убедитесь, что вы:
- Добились Recall ≥ 0.80.
- Код в ноутбуке выполняется последовательно (Run All) без ошибок.
- Присутствует текстовый вывод с итогами сравнения моделей.
- Сделали копию ноутбука доступной для просмотра (если выполняли задание в Google Colab).